# ARC-AGI Latent Program JEPA (LP-JEPA)

In [1]:
import json, math, os, random, copy, glob
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)

device: cuda | torch: 2.8.0+cu128


In [2]:
import subprocess, os
ARC_DIR = Path("data/arc")
if not ARC_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/fchollet/ARC-AGI.git", str(ARC_DIR)],
        check=True,
    )
print("train tasks:", len(list((ARC_DIR / "data/training").glob("*.json"))))
print("eval  tasks:", len(list((ARC_DIR / "data/evaluation").glob("*.json"))))

train tasks: 400
eval  tasks: 400


In [3]:
def load_arc_split(split):
    """Return dict task_id -> {'train': [(in,out),...], 'test': [(in,out),...]}."""
    out = {}
    for p in sorted((ARC_DIR / "data" / split).glob("*.json")):
        d = json.loads(p.read_text())
        out[p.stem] = {
            "train": [(np.array(e["input"]), np.array(e["output"])) for e in d["train"]],
            "test":  [(np.array(e["input"]), np.array(e["output"])) for e in d["test"]],
        }
    return out

TRAIN_TASKS = load_arc_split("training")
EVAL_TASKS  = load_arc_split("evaluation")
print(len(TRAIN_TASKS), len(EVAL_TASKS))

400 400


In [4]:
assert len(TRAIN_TASKS) == 400 and len(EVAL_TASKS) == 400
g = next(iter(TRAIN_TASKS.values()))["train"][0][0]
assert g.ndim == 2 and g.min() >= 0 and g.max() <= 9
print("OK: 400/400 tasks, grids are 2D int 0-9")

OK: 400/400 tasks, grids are 2D int 0-9
